In [1]:
import os
import sys
import boto3
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))


In [ ]:
from research_assistant.src.agents.paper_searcher import SearchAgent
from src.services.paper_search import paper_search_tool


In [3]:
SEARCH_MODEL = "meta.llama3-8b-instruct-v1:0"
user_query = "What is the role of AMP-activated protein kinase (AMPK) in metabolic disorders such as type 2 diabetes?"

In [ ]:
search_agent = SearchAgent(
    llm_id = SEARCH_MODEL,
    search_tool = paper_search_tool
)
question = user_query
papers = search_agent.run(question, limit=10)

papers


In [55]:
import json
#with open("raw_papers_list.json", "w") as f:
#    json.dump(papers, f)

with open("raw_papers_list.json", "r") as f:
    papers = json.load(f)

In [56]:
papers

[{'id': 'e2539e95ef932c51ed0e1653528038873755fe85',
  'title': 'Canagliflozin promotes osteoblastic MC3T3-E1 differentiation via AMPK/RUNX2 and improves bone microarchitecture in type 2 diabetic mice',
  'year': 2022,
  'venue': 'Frontiers in Endocrinology',
  'fieldsOfStudy': ['Medicine'],
  'publicationTypes': ['JournalArticle'],
  'pdf_url': 'https://www.frontiersin.org/articles/10.3389/fendo.2022.1081039/pdf'},
 {'id': '0d0f4c28e0995d6c98ef73c205d3d57a8280389a',
  'title': 'Molecular basis of AR and STK11 genes associated pathogenesis via AMPK pathway and adipocytokine signalling pathway in the development of metabolic disorders in PCOS women',
  'year': 2022,
  'venue': 'Beni-Suef University Journal of Basic and Applied Sciences',
  'fieldsOfStudy': None,
  'publicationTypes': None,
  'pdf_url': 'https://bjbas.springeropen.com/track/pdf/10.1186/s43088-022-00200-8'},
 {'id': '03edfe1dd1bf7399a40ee5a4854e82841e8d0531',
  'title': 'Ginsenoside Rg1 inhibits dietary-induced obesity and

In [5]:
from src.services.paper_ingestion import PaperIngestionService

In [ ]:
papers_text = []
ingestion = PaperIngestionService()
for paper in papers:
    try:
        paper["text"] = ingestion.get_text(paper)
        papers_text.append(paper)
    except Exception as e:
        print(f"Error processing paper {paper.get('id', 'unknown')}: {e}")
        papers_text.append({})

papers_text #list of dictionaries

Skipping non-PDF file: http://www.scielo.br/pdf/bjmbr/v51n4/1414-431X-bjmbr-51-4-e7139.pdf
Error processing paper 03edfe1dd1bf7399a40ee5a4854e82841e8d0531: An error occurred while processing the PDF: 03edfe1dd1bf7399a40ee5a4854e82841e8d0531. Error: 'NoneType' object does not support the context manager protocol
Error processing paper de4a103e26a1a52ac0f6a07f88a93baf96604c26: Failed to download the PDF from the URL: https://www.mdpi.com/1422-0067/25/1/453/pdf?version=1703830534. Error: 403 Client Error: Forbidden for url: https://www.mdpi.com/1422-0067/25/1/453/pdf?version=1703830534
Error processing paper a2b2a9be0ff7ca3a6a1e9816f76e6df10b218ddc: Failed to download the PDF from the URL: https://www.mdpi.com/1424-8247/14/3/238/pdf?version=1615359278. Error: 403 Client Error: Forbidden for url: https://www.mdpi.com/1424-8247/14/3/238/pdf?version=1615359278
Error processing paper 6dc88866a109ca5bf10aff1be55ee4bd25fa37bd: Failed to download the PDF from the URL: https://www.nature.com/arti

[{'id': 'e2539e95ef932c51ed0e1653528038873755fe85',
  'title': 'Canagliflozin promotes osteoblastic MC3T3-E1 differentiation via AMPK/RUNX2 and improves bone microarchitecture in type 2 diabetic mice',
  'year': 2022,
  'venue': 'Frontiers in Endocrinology',
  'fieldsOfStudy': ['Medicine'],
  'publicationTypes': ['JournalArticle'],
  'pdf_url': 'https://www.frontiersin.org/articles/10.3389/fendo.2022.1081039/pdf',
  'text': 'Canagliﬂozin promotes\nosteoblastic MC3T3-E1\ndifferentiation via AMPK/RUNX2\nand improves bone\nmicroarchitecture in type 2\ndiabetic mice\nPeiyang Song1† , TianyiChen1† , ShunliRui1, XiaodongDuan2,\nBo Deng1, David G.Armstrong3,Y uMa1* and WuquanDeng1*\n1Department of Endocrinology, Chongqing Emergency Medical Center, Chongqing University\nCentral Hospital, School of Medicine, Chongqing University, Chongqing, China,2Department of\nRehabilitation, The Afﬁliated Hospital of Southwest Medical University, Luzhou, Sichuan, China,\n3Department of Surgery, Keck School o

In [11]:
with open("text_papers_list.json", "w") as f:
    json.dump(papers_text, f)

In [26]:
import json

STRENGTH_RANK = {
    "weak": 1,
    "moderate": 2,
    "strong": 3,
}

class ClaimsService:
    def __init__(self, claims_agent):
        self.claims_agent = claims_agent

        
    def _clean_json(self, json_text:str) -> list:
        cleaned_json = json_text.strip()
        cleaned_json = cleaned_json.removeprefix("```json").removesuffix("```").strip()
        try:
            return json.loads(cleaned_json)
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")
            return [] 
    

    def process_chunks(self, chunks:list):
        if not chunks:
            print("Error, no chunks")
        
        chunk_outputs = []
        for chunk in chunks:
            claims = self.claims_agent.run_claim_extractor(chunk["paper_id"], chunk["chunk_id"], chunk["text"])
            if not claims:
                continue
            claims = self._clean_json(claims)
            claims_dict = {
                "paper_id": chunk.get("paper_id", ""),
                "chunk_id": chunk.get("chunk_id", ""),
                "claims": claims
            }
            chunk_outputs.append(claims_dict)

        return chunk_outputs
    
    
    def aggregate_claims(self, chunk_outputs: list, paper_id: str):
        """
        Aggregates claims extracted from multiple chunks of the same paper.

        Args:
            chunk_outputs (list): List of dicts, each containing:
                - chunk_id
                - claims (list of claim dicts)
            paper_id (str): Paper identifier

        Returns:
            dict: Aggregated paper-level claims
        """

        aggregated = {}

        for chunk in chunk_outputs:
            chunk_id = chunk["chunk_id"]
            claims = chunk.get("claims", [])

            for c in claims:
                claim_text = c["claim"].strip()

                if claim_text not in aggregated:
                    aggregated[claim_text] = {
                        "claim": claim_text,
                        "evidence": set(),
                        "sections": set(),
                        "strength": c.get("strength", "moderate"),
                        "source_chunks": set(),
                    }

                aggregated_entry = aggregated[claim_text]

                aggregated_entry["evidence"].add(c.get("evidence", "").strip())
                aggregated_entry["sections"].add(c.get("section", "unknown"))
                aggregated_entry["source_chunks"].add(chunk_id)

                # Upgrade strength if stronger evidence appears
                current_strength = aggregated_entry["strength"]
                new_strength = c.get("strength", "moderate")

                if STRENGTH_RANK.get(new_strength, 0) > STRENGTH_RANK.get(
                    current_strength, 0
                ):
                    aggregated_entry["strength"] = new_strength

        # Convert sets to lists for JSON compatibility
        final_claims = []
        for entry in aggregated.values():
            final_claims.append(
                {
                    "claim": entry["claim"],
                    "evidence": list(entry["evidence"]),
                    "sections": list(entry["sections"]),
                    "strength": entry["strength"],
                    "source_chunks": list(entry["source_chunks"]),
                }
            )

        return final_claims

In [9]:
def chunk_paper(
    paper_id: str,
    full_text: str,
    max_chars: int = 16_000,
    overlap_chars: int = 1_000,
):
    """
    Splits a paper's text into overlapping character-based chunks,
    suitable for Claude models (no tokenizer required).

    Args:
        paper_id (str): Unique identifier for the paper
        full_text (str): Cleaned full text of the paper
        max_chars (int): Maximum characters per chunk
        overlap_chars (int): Overlapping characters between chunks

    Returns:
        List[dict]: Each dict contains paper_id, chunk_id, chunk_index, text
    """

    if not full_text or not full_text.strip():
        return []

    if overlap_chars >= max_chars:
        raise ValueError("overlap_chars must be smaller than max_chars")

    chunks = []
    text_length = len(full_text)
    start = 0
    chunk_index = 0

    while start < text_length:
        end = min(start + max_chars, text_length)
        chunk_text = full_text[start:end]
        chunks.append(
            {
                "paper_id": paper_id,
                "chunk_id": f"{paper_id}_chunk_{chunk_index:03d}",
                "chunk_index": chunk_index,
                "text": chunk_text,
            }
        )
        chunk_index += 1

        if end == text_length:
            break
        start = end - overlap_chars

    return chunks

In [10]:
import boto3

BRT = boto3.client("bedrock-runtime")
SYSTEM_PROMPT = (
    "You are an expert scientific research analyst.\n\n"
    "Your task is to extract explicit empirical claims from academic research papers.\n"
    "A claim must be directly supported by evidence in the provided text.\n\n"
    "Rules:\n"
    "- Focus only on results, findings, outcomes, or conclusions\n"
    "- Ignore background, literature review, methods, and references\n"
    "- Ignore speculative or future work statements\n"
    "- Do not hallucinate or infer beyond the text\n"
    "- If no extractable claims exist, return an empty list\n\n"
    
    "Output rules(STRICT):\n"
    "- Output ONLY a valid JSON array with at most 5 claim objects\n"
    "- Do NOT use markdown\n"
    "- Do NOT include explanations or comments.\n"
    "- Do NOT include paper_id or chunk identifiers.\n"
    "- Escape all quotation marks inside strings.\n"
    "- If the output would be truncated, return an empty JSON array instead"

    "- Each array element MUST match this schema:\n\n"
    "{\n"
    '   "claim": "string", \n'
    '   "evidence": "string",\n'
    '   "section": "string",\n'
    '   "strength": "strong | moderate | weak"\n'
    "}"
    
)



class ClaimExtractor:
    def __init__(self, model_id: str = "global.anthropic.claude-haiku-4-5-20251001-v1:0", max_output_tokens: int = 900):
        self.model_id = model_id
        self.max_output_tokens = max_output_tokens


    def build_user_prompt(self, text) -> str:
        return f"""
    You are given a chunk of text from a research paper.
    Text:
    \"\"\"
    {text}
    \"\"\"
    Extract all empirical claims present in this text.
    """.strip()


    def run_claim_extractor(self, paper_id:str, chunk_id:str, text: str):
        conversation = [
            {
                "role": "user",
                "content": [{"text": self.build_user_prompt(text)}],
            },
        ]

        response = BRT.converse(
            modelId=self.model_id,
            system=[
                {
                    "text": SYSTEM_PROMPT
                }
            ],
            messages=conversation,
            inferenceConfig={
                "maxTokens": self.max_output_tokens,
                "temperature": 0.0,
            },
        )

        stop_reason = response.get("stopReason", "")
        if stop_reason == "max_tokens":
            print(f"Output truncated for paper_id: {paper_id}, chunk_id:{chunk_id}")
            return[]

        output_text = response["output"]["message"]["content"][0]["text"]

        return output_text


In [ ]:
claim_agent = ClaimExtractorAgent()
claim_service = ClaimsService(claim_agent)

In [28]:
all_papers_claims = []

for paper in papers_text:
    if not paper:
        continue
    chunks = chunk_paper(paper["id"], paper["text"])
    chunks_claims = claim_service.process_chunks(chunks)
    paper_claims = claim_service.aggregate_claims(chunks_claims, paper["id"])
    claimed_paper = {
    "paper_id":paper["id"],
    "metadata":{
        "title": paper["title"],
        "year": paper["year"],
        "venue": paper["venue"],
        "field_of_study": paper["fieldsOfStudy"],
        "publication_type": paper["publicationTypes"]
    },
    "claims": paper_claims
    }
    all_papers_claims.append(claimed_paper)
        
    

Output truncated for paper_id: 718e1e47027b6b851d5d01bd3c956ead1a7e2e3e, chunk_id:718e1e47027b6b851d5d01bd3c956ead1a7e2e3e_chunk_001


In [29]:
all_papers_claims

[{'paper_id': 'e2539e95ef932c51ed0e1653528038873755fe85',
  'metadata': {'title': 'Canagliflozin promotes osteoblastic MC3T3-E1 differentiation via AMPK/RUNX2 and improves bone microarchitecture in type 2 diabetic mice',
   'year': 2022,
   'venue': 'Frontiers in Endocrinology',
   'field_of_study': ['Medicine'],
   'publication_type': ['JournalArticle']},
  'claims': [{'claim': 'Canagliflozin increased bone mineral density in type 2 diabetic mice',
    'evidence': ['canagliflozin, but not dapagliflozin or empagliflozin, increased bone mineral density (p<0.05) and improved bone microarchitecture in type 2 diabetic mice'],
    'sections': ['Results/Abstract'],
    'strength': 'strong',
    'source_chunks': ['e2539e95ef932c51ed0e1653528038873755fe85_chunk_000']},
   {'claim': 'Canagliflozin promoted osteoblast differentiation at 5mM concentration under high glucose conditions',
    'evidence': ['canagliflozin promoted osteoblast differentiation at a concentration of 5mM under high glucos

In [30]:
with open("all_claims.json", "w") as f:
    json.dump(all_papers_claims, f)

### study quality evaluator 

In [ ]:
BRT = boto3.client("bedrock-runtime")
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"


class QualityEvaluator:
    def __init__(self):
        pass
    
    def build_user_prompt(self, data: dict) -> str:
        return f"""
    You are a scientific study quality assessment agent.

    Your role is to evaluate the reliability of a research study based strictly on the provided claims and metadata.

    Data:
    \"\"\"
    {data}
    \"\"\"

    Guidelines:
    - Do not infer methods or details that are not explicitly supported by the provided information.
    - If any information is missing or unclear, explicitly mark it as "unclear."
    - Be conservative in your assessment and avoid overconfidence in your conclusions.

    Output:
    - Return ONLY a valid JSON object that adheres to the following schema:
    {{
    "paper_id": "...",
    "study_quality": {{
        "study_design": "RCT | observational | cohort | meta-analysis | unclear",
        "sample_size": "large | medium | small | unclear",
        "bias_risk": "low | moderate | high",
        "evidence_strength": "strong | moderate | weak",
        "notes": "short justification grounded in claims"
    }}
    }}

    Strict Rules:
    - Do NOT include any additional text, explanations, or comments outside the JSON object.
    - Ensure the JSON is well-formed and adheres to the schema exactly.
    """.strip()


    def evaluate_study_quality(self, data:dict):
        conversation = [
        {
            "role": "user",
            "content": [{"text":self.build_user_prompt(data)}]
        }
        ]
        response = BRT.converse(
            modelId= MODEL_ID,
            messages=conversation
            # add more config
        )

        raw_output = response["output"]["message"]["content"][0]["text"]
        return raw_output

        

In [ ]:
def clean_json(json_text:str) -> list:
    cleaned_json = json_text.strip()
    cleaned_json = cleaned_json.removeprefix("```json").removesuffix("```").strip()
    try:
        return json.loads(cleaned_json)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        return []
    

def normalize_enum(value, allowed, default="unclear"):
    if not value:
        return default
    value = value.strip().lower()
    return value if value in allowed else default


ALLOWED_STUDY_DESIGN = {
    "rct", "observational", "cohort", "meta-analysis", "unclear"
}

ALLOWED_SAMPLE_SIZE = {
    "large", "medium", "small", "unclear"
}

ALLOWED_BIAS_RISK = {
    "low", "moderate", "high"
}

ALLOWED_EVIDENCE_STRENGTH = {
    "strong", "moderate", "weak"
}


def validate_study_quality(raw_output: dict) -> dict:
    sq = raw_output.get("study_quality", {})

    validated = {
        "study_design": normalize_enum(
            sq.get("study_design"),
            ALLOWED_STUDY_DESIGN
        ),
        "sample_size": normalize_enum(
            sq.get("sample_size"),
            ALLOWED_SAMPLE_SIZE
        ),
        "bias_risk": normalize_enum(
            sq.get("bias_risk"),
            ALLOWED_BIAS_RISK,
            default="high"   # conservative default
        ),
        "evidence_strength": normalize_enum(
            sq.get("evidence_strength"),
            ALLOWED_EVIDENCE_STRENGTH,
            default="weak"
        ),
        "notes": sq.get("notes", "").strip()[:500]
    }

    return {
        "paper_id": raw_output.get("paper_id"),
        "study_quality": validated
    }


def attach_quality_to_claims(
    claims: list[dict],
    study_quality_result: dict,
) -> list[dict]:
    """
    Attaches paper-level study quality metadata to each claim.

    Args:
        claims (list[dict]): Claims extracted from a single paper
        study_quality_result (dict): Output of study quality agent

    Returns:
        list[dict]: Claims enriched with study_quality
    """

    if not claims:
        return []

    # Fail-closed default quality
    default_quality = {
        "study_design": "unclear",
        "sample_size": "unclear",
        "bias_risk": "high",
        "evidence_strength": "weak",
        "notes": ""
    }

    quality = study_quality_result.get("study_quality", default_quality)

    enriched_claims = []

    for claim in claims:
        enriched_claim = {
            **claim,
            "study_quality": quality
        }
        enriched_claims.append(enriched_claim)

    return enriched_claims

In [ ]:
evaluator = QualityEvaluatorAgent()
evaluated_papers = []
for paper in all_papers_claims:
    evaluated = evaluator.evaluate_study_quality(paper)
    validated = validate_study_quality(evaluated)
    enriched_claims = attach_quality_to_claims(paper["claims"], validated)
    updated_claims = [{**claim, "paper_id":paper['paper_id'], "title":paper['metadata']['title']} for claim in enriched_claims]
    evaluated_papers.append(updated_claims)
    

```json
{
  "paper_id": "e2539e95ef932c51ed0e1653528038873755fe85",
  "study_quality": {
    "study_design": "unclear",
    "sample_size": "unclear",
    "bias_risk": "moderate",
    "evidence_strength": "strong",
    "notes": "Study combines in vivo (type 2 diabetic mice) and in vitro (MC3T3-E1 cells) experiments with mechanistic investigations. Strong evidence from multiple claims supported by statistical significance (p<0.05 or better), controlled comparisons between drug treatments, and pathway validation using inhibitors/activators. Study design details, specific sample sizes, and animal numbers are not provided in claims. Moderate bias risk due to: (1) lack of blinding information, (2) potential publication bias toward positive results for canagliflozin, (3) in vitro findings may not translate to in vivo mechanisms. Differential effects between SGLT2 inhibitors suggest specificity but warrant independent replication."
  }
}
```
```json
{
  "paper_id": "0d0f4c28e0995d6c98ef73c205d

In [39]:
evaluated_papers

[[{'claim': 'Canagliflozin increased bone mineral density in type 2 diabetic mice',
   'evidence': ['canagliflozin, but not dapagliflozin or empagliflozin, increased bone mineral density (p<0.05) and improved bone microarchitecture in type 2 diabetic mice'],
   'sections': ['Results/Abstract'],
   'strength': 'strong',
   'source_chunks': ['e2539e95ef932c51ed0e1653528038873755fe85_chunk_000'],
   'study_quality': {'study_design': 'unclear',
    'sample_size': 'unclear',
    'bias_risk': 'moderate',
    'evidence_strength': 'strong',
    'notes': 'Study combines in vivo (type 2 diabetic mice) and in vitro (MC3T3-E1 cells) experiments with mechanistic investigations. Strong evidence from multiple claims supported by statistical significance (p<0.05 or better), controlled comparisons between drug treatments, and pathway validation using inhibitors/activators. Study design details, specific sample sizes, and animal numbers are not provided in claims. Moderate bias risk due to: (1) lack of 

## embedding

In [ ]:
# download and read index and metadata
import faiss
import pickle

BUCKET = "researchassistant-rag-prod-embeddings"
INDEX_PATH = "claims_test.index"
META_PATH = "meta_test.pkl"
EMBED_MODEL_ID = "amazon.titan-embed-text-v2:0"

session = boto3.Session(profile_name="aws_learning")
s3 = session.client("s3", region_name="ca-central-1")

downloaded_index_path = "downloads/faiss_claims.index"
downloaded_metadata_path = "downloads/metadata.pkl"

s3.download_file(BUCKET, Key = "faiss/claims.index", Filename= downloaded_index_path )
s3.download_file(BUCKET, "faiss/claims_meta.pkl", downloaded_metadata_path)

downloaded_index = faiss.read_index(downloaded_index_path)
with open(downloaded_metadata_path, "rb") as f:
    downloaded_metadata = pickle.load(f)

In [43]:
print(len(downloaded_metadata))
print(downloaded_index.ntotal)

9
9


In [ ]:
bedrock_client = session.client(
    service_name="bedrock-runtime",
    region_name="ca-central-1",
)

embed = Embedding(
    EMBED_MODEL_ID,
    bedrock_client,
    downloaded_index,
    downloaded_metadata

)

for paper in evaluated_papers:
    embed.append_claims(paper)


In [53]:
print(len(downloaded_metadata))
print(downloaded_index.ntotal)

24
24
